In [1]:
print("hello")

hello


In [7]:
import cv2
import numpy as np
from pathlib import Path

In [8]:
from pathlib import Path

IMG_DIR = Path("mouse_dataset/images_perfect")

num_files = sum(1 for f in IMG_DIR.iterdir() if f.is_file())

print(f"Number of files: {num_files}")

Number of files: 1398


In [ ]:
from pathlib import Path

IMG_DIR = Path("mouse_dataset/images_mgs_crop_manual")

files = sorted(
    [f.name for f in IMG_DIR.glob("*.jpg")]
)
# 011742
target = "023149.jpg"

position = files.index(target) + 1

print(f"{target} is image #{position} out of {len(files)}")

023149.jpg is image #2000 out of 3011


In [2]:
def crop_inside_black_rectangle(image_path, output_path, margin=8):
    img = cv2.imread(str(image_path))
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    # 只找非常黑的像素
    mask = cv2.inRange(gray, 0, 45)

    # 找轮廓，用 RETR_LIST 而不是 RETR_EXTERNAL
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_LIST,
        cv2.CHAIN_APPROX_SIMPLE
    )

    candidates = []

    H, W = gray.shape

    for c in contours:
        x, y, w, h = cv2.boundingRect(c)

        # 过滤太小的区域
        if w < 100 or h < 100:
            continue

        # 过滤整张图外边框
        if w > 0.95 * W or h > 0.95 * H:
            continue

        area = w * h
        candidates.append((x, y, w, h, area))

    if not candidates:
        raise ValueError("No black rectangle found")

    # 选面积最大的内部矩形
    x, y, w, h, _ = max(candidates, key=lambda b: b[4])

    # 去掉黑色边框本身，只保留框内内容
    crop = img[
        y + margin : y + h - margin,
        x + margin : x + w - margin
    ]

    cv2.imwrite(str(output_path), crop)

    return x, y, w, h

In [4]:
x,y,w,h=crop_inside_black_rectangle(
    "000008.jpg",
    "000008_crop.jpg"
)

## 剪裁+高斯模糊

In [5]:
import cv2
import numpy as np
from pathlib import Path


def create_muzzle_and_blur_ablation(
    boxed_image_path,
    original_image_path,
    muzzle_output,
    blur_output,
    margin=15,
    blur_kernel=301
):
    """
    输入:
        boxed_image_path:
            带黑框的图片，用来检测 muzzle box 坐标

        original_image_path:
            原始无黑框图片，用来生成最终图片

    输出:
        1. muzzle_output:
            黑框内部裁剪出的 muzzle-only 图片

        2. blur_output:
            原图中 muzzle 区域被高斯模糊后的图片，没有黑框
    """

    # =========================
    # 读取带黑框图片，用于检测坐标
    # =========================

    boxed_img = cv2.imread(str(boxed_image_path))

    if boxed_img is None:
        raise ValueError(f"Cannot read boxed image: {boxed_image_path}")

    gray = cv2.cvtColor(boxed_img, cv2.COLOR_BGR2GRAY)

    # =========================
    # 找黑框
    # =========================

    mask = cv2.inRange(gray, 0, 45)

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_LIST,
        cv2.CHAIN_APPROX_SIMPLE
    )

    H, W = gray.shape
    candidates = []

    for c in contours:
        x, y, w, h = cv2.boundingRect(c)

        if w < 100 or h < 100:
            continue

        # 排除整张图外边缘
        if w > 0.95 * W or h > 0.95 * H:
            continue

        area = w * h
        candidates.append((x, y, w, h, area))

    if len(candidates) == 0:
        raise ValueError(f"No black rectangle found: {boxed_image_path}")

    x, y, w, h, _ = max(candidates, key=lambda b: b[4])

    # =========================
    # 黑框内部区域，去掉边框线
    # =========================

    x1 = x + margin
    y1 = y + margin
    x2 = x + w - margin
    y2 = y + h - margin

    # 防止越界
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(W, x2)
    y2 = min(H, y2)

    if x2 <= x1 or y2 <= y1:
        raise ValueError("Invalid crop region after applying margin")

    # =========================
    # 读取原始无黑框图片
    # =========================

    original_img = cv2.imread(str(original_image_path))

    if original_img is None:
        raise ValueError(f"Cannot read original image: {original_image_path}")

    if original_img.shape[:2] != boxed_img.shape[:2]:
        raise ValueError(
            "Original image and boxed image must have the same width and height"
        )

    # =========================
    # 1. Muzzle Only
    # 从原始图片裁剪，而不是从带黑框图片裁剪
    # =========================

    muzzle_crop = original_img[y1:y2, x1:x2]

    cv2.imwrite(
        str(muzzle_output),
        muzzle_crop
    )

    # =========================
    # 2. Blur Ablation
    # 在原始图片上模糊 muzzle 区域，所以不会保留黑框
    # =========================

    blur_img = original_img.copy()

    roi = blur_img[y1:y2, x1:x2]

    if blur_kernel % 2 == 0:
        blur_kernel += 1

    blurred_roi = cv2.GaussianBlur(
        roi,
        (blur_kernel, blur_kernel),
        0
    )

    blur_img[y1:y2, x1:x2] = blurred_roi

    cv2.imwrite(
        str(blur_output),
        blur_img
    )

    return {
        "bbox": (x1, y1, x2, y2),
        "muzzle_size": muzzle_crop.shape,
        "blur_kernel": blur_kernel
    }

In [7]:


result = create_muzzle_and_blur_ablation(
    boxed_image_path="000008.jpg",
    original_image_path="000008.jpg",
    muzzle_output="000008_crop.jpg",
    blur_output="000008_blur.jpg",
    margin=15,
    blur_kernel=301
)

print(result)

{'bbox': (163, 15, 1003, 941), 'muzzle_size': (926, 840, 3), 'blur_kernel': 301}
